# 11 · Adaptation

The **dynamic** model-free readout: when the distribution shifts, does the PSE move
toward the new optimum, and how does that unfold **session by session**.

**Per-session, not pooled.** Adaptation here is a within-session process, so each
hard session is its own trajectory — no rolling window crosses a session boundary,
and sessions are never glued into one stream. This matters because the two cohorts
have different structure: SS01–13 run *several* Hard-A then Hard-B sessions (a run,
where overnight reset vs carry-over is visible); SS14–23 have *one* session per hard
phase in an ABAB chain. A multi-session plateau is meaningless for a single session,
so it is retired — the scalars are per-session **start offset** and **end offset**.

**All trials, including opto.** The question is silencing's *cumulative* effect on a
session's learning (does adaptation take longer overall), so on-trials belong in —
the within-trial on/off split is 2X's job.

**Baseline = expert uniform.** One fixed origin per animal: the pooled PSE of the
last 5 `expert_uniform` sessions. Every value is `PSE(t) − pse_expert`, so 0 is
genuine uniform behaviour — you can see whether a session starts back near uniform
(overnight reset) or where the last one left off. The dashed line is the normative
target (`pse_normative − pse_expert`). Cohort-split as in 10.

In [ ]:
from shared_setup import *
apply_style()

experiment, info = load_data()
first_ids = [a for a in FIRST_COHORT if a in experiment.animals]
opto_ids  = [a for a in OPTO_COHORT  if a in experiment.animals]
print(f"first ({len(first_ids)}): {first_ids}")
print(f"opto  ({len(opto_ids)}): {opto_ids}")

In [ ]:
TO_DISTS = ['Hard-A', 'Hard-B']
TRIALS   = 'all'          # silencing's cumulative effect stays in; 'non_opto' to exclude
WINDOW, STEP = 50, 10

def adapt(animal, to_dist):
    try:
        return compute_adaptation_per_session(animal, to_dist, trials=TRIALS,
                                              window=WINDOW, step=STEP)
    except Exception:
        return None

## §2 · One animal, end to end

A first-cohort animal (the rich case: a run of hard sessions). Per switch
distribution, its sessions laid out in order — each a separate trajectory segment
(boundaries by construction), against the expert-uniform (0) and normative (dashed)
lines. The opto cohort's single-session shape shows up in §4.

In [ ]:
ex_id = first_ids[0] if first_ids else opto_ids[0]
ex = experiment.get_animal(ex_id)

fig, axes = plt.subplots(len(TO_DISTS), 1, figsize=(9, 3.4 * len(TO_DISTS)), squeeze=False)
for ax, to_dist in zip(axes[:, 0], TO_DISTS):
    r = adapt(ex, to_dist)
    if r is None or not r['sessions']:
        ax.set_title(f"{to_dist}: no sessions"); continue
    plot_adaptation_sessions(r, ax=ax, layout='concat')
    ax.set_title(f"{ex_id} → {to_dist}  ({r['meta']['n_sessions']} sessions, pse_expert={r['pse_expert']:.2f})", fontsize=10)
fig.tight_layout()

## §3 · Fold — per session

Every animal, every hard session: **start_offset** and **end_offset** (first- and
last-window `PSE − pse_expert`) → tidy `{animal, cohort, to_dist, session_id,
switch_index, stat, value}`. Kept at the per-session grain (richer representation) —
collapse to a per-animal summary downstream when the question is fixed.

`switch_index` = which appearance of the distribution a session is (0 = first). For
the opto cohort's ABAB chain this is 0, 1, 2, … so early-vs-late switches stay
recoverable; for the first cohort a whole run shares one index.

In [ ]:
persession_rows, results = [], {}
for coh, ids in [('first', first_ids), ('opto', opto_ids)]:
    for aid in ids:
        animal = experiment.get_animal(aid)
        for to_dist in TO_DISTS:
            r = adapt(animal, to_dist)
            results[(aid, to_dist)] = r
            if r is None:
                continue
            for row in r['rows']:
                persession_rows.append({'animal': aid, 'cohort': coh, 'to_dist': to_dist,
                                        'session_id': row['session_id'],
                                        'switch_index': row['switch_index'],
                                        'stat': row['stat'], 'value': row['value']})
persession_df = pd.DataFrame(persession_rows)
print(f"{persession_df['animal'].nunique()} animals, "
      f"{persession_df[['animal','to_dist','session_id']].drop_duplicates().shape[0]} sessions, "
      f"{len(persession_df)} rows")
persession_df.head()

## §4 · Cohort views

Two questions, descriptive, within cohort. **Offset vs switch** — does where a
session starts (overnight reset if ≈ 0) and where it ends (adaptation reached) change
across successive switches? For the opto ABAB chain, whether end_offset weakens at
later switch_index is one thing to look at (not something this data can confirm).
**First-session
trajectories** — each animal's first appearance, overlaid, re-zeroed.

In [1]:
# offset vs switch_index, one line per animal (start and end), per cohort x to_dist
for coh, ids in [('first', first_ids), ('opto', opto_ids)]:
    for to_dist in TO_DISTS:
        sub = persession_df[(persession_df['cohort'] == coh) & (persession_df['to_dist'] == to_dist)]
        if sub.empty: continue
        fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
        for ax, stat in zip(axes, ['start_offset', 'end_offset']):
            s = sub[sub['stat'] == stat]
            for aid, g in s.groupby('animal'):
                g = g.sort_values('switch_index')
                ax.plot(g['switch_index'], g['value'], marker='o', ms=4, alpha=0.7, label=aid)
            ax.axhline(0.0, color='0.6', ls='--', lw=1)
            ax.set_xlabel('switch_index'); ax.set_title(stat, fontsize=10)
        axes[0].set_ylabel('PSE − expert')
        fig.suptitle(f"{coh} cohort — {to_dist}: offset across switches", fontsize=12); fig.tight_layout()

NameError: name 'first_ids' is not defined

In [ ]:
# first-session (switch_index 0) trajectories overlaid, re-zeroed, per cohort x to_dist
for coh, ids in [('first', first_ids), ('opto', opto_ids)]:
    fig, axes = plt.subplots(1, len(TO_DISTS), figsize=(4.5 * len(TO_DISTS), 3.4), squeeze=False, sharey=True)
    for ax, to_dist in zip(axes[0], TO_DISTS):
        n = 0
        for aid in ids:
            r = results.get((aid, to_dist))
            if not r: continue
            for e in r['sessions']:
                if e['switch_index'] == 0 and len(e['trials']):
                    x = np.asarray(e['trials'], dtype=float)
                    ax.plot(x - x[0], e['values'], color='0.4', alpha=0.4, lw=1.2); n += 1
        ax.axhline(0.0, color='0.6', ls='--', lw=1)
        ax.set_xlabel('trials since session start'); ax.set_title(f"{to_dist} (n={n})", fontsize=10)
    axes[0][0].set_ylabel('PSE − expert')
    fig.suptitle(f"{coh} cohort — first-session adaptation", fontsize=12); fig.tight_layout()

## §5 · Summary + emit

Per-session offsets, at the per-session grain, for reference and 2X. Read-only.

In [ ]:
# try:
#     out = FIG_DIR.parent / 'results'; out.mkdir(parents=True, exist_ok=True)
#     persession_df.to_csv(out / '11_adaptation_per_session.csv', index=False)
#     print(f'wrote per-session adaptation ({len(persession_df)} rows) to {out}')
# except Exception as e:
#     print(f'emit skipped: {e}')